# Face Emotion Recognition

https://huggingface.co/datasets/tukey/human_face_emotions_roboflow/viewer/default/train?p=1&views%5B%5D=train

# Import Data

In [2]:
import pandas as pd
import io
from PIL import Image

df = pd.read_parquet("hf://datasets/tukey/human_face_emotions_roboflow/data/train-00000-of-00001.parquet")

c:\Users\rmaia\.conda\envs\roxsenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Overview & Cleaning

In [3]:
# Standardize column names (strip whitespace, lower-case, replace spaces with underscores)
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

# Check for missing values
print("Missing values per column:")
print(df.isna().sum())

# No missing values or duplicates, so we can proceed with the data as is

# Print out summary information
print("\nDataframe Info:")
print(df.info())

# Print the first few rows to inspect the data
print("\nFirst 5 rows of the dataset:")
print(df.head())

Missing values per column:
image    0
qa       0
dtype: int64

Dataframe Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9400 entries, 0 to 9399
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   image   9400 non-null   object
 1   qa      9400 non-null   object
dtypes: object(2)
memory usage: 147.0+ KB
None

First 5 rows of the dataset:
                                               image  \
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...   

                                                  qa  
0  [{'question': 'How does the person feel in the...  
1  [{'question': 'How does the person feel in the...  
2  [{'question': 'How does the person feel in the...  
3  [{'question': 'How does the person

In [4]:
# Check for missing values in each column
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
image    0
qa       0
dtype: int64


In [5]:
import json
import numpy as np

def extract_emotion(qa_entry):
    try:
        # If the qa_entry is a string, strip it and parse as JSON.
        if isinstance(qa_entry, str):
            qa_entry = qa_entry.strip()
            qa_data = json.loads(qa_entry)
        else:
            qa_data = qa_entry

        # If the data is a numpy array, convert it to a list.
        if isinstance(qa_data, np.ndarray):
            qa_data = qa_data.tolist()

        # Now you can check if it's a list or tuple using this condition.
        if isinstance(qa_data, (list, tuple)) and len(qa_data) > 0:
            return qa_data[0].get("answer")
        else:
            print("Unexpected qa_data structure:", qa_data, "with type", type(qa_data))
    except Exception as e:
        print("Error parsing qa entry:", qa_entry, "\nError:", e)
    return None

In [6]:
# Assuming df is your DataFrame that includes the 'qa' column
df["emotion"] = df["qa"].apply(extract_emotion)

# Verify the new column
print(df[["qa", "emotion"]].head())

                                                  qa  emotion
0  [{'question': 'How does the person feel in the...      sad
1  [{'question': 'How does the person feel in the...    anger
2  [{'question': 'How does the person feel in the...  neutral
3  [{'question': 'How does the person feel in the...     fear
4  [{'question': 'How does the person feel in the...  content


In [7]:
# Check unique values and distribution of facial emotion labels
if 'emotion' in df.columns:
    print("\nUnique emotion labels:")
    print(df['emotion'].unique())

    print("\nDistribution of emotion labels:")
    print(df['emotion'].value_counts())


Unique emotion labels:
['sad' 'anger' 'neutral' 'fear' 'content' 'happy' 'disgust' 'surprise']

Distribution of emotion labels:
emotion
surprise    1238
neutral     1225
sad         1184
fear        1181
anger       1175
disgust     1165
content     1144
happy       1088
Name: count, dtype: int64


In [8]:
import matplotlib.pyplot as plt

# Example: Plot a histogram for a numeric column, adjust 'score' to the relevant column name
if 'score' in df.columns:
    plt.hist(df['score'].dropna(), bins=30, edgecolor='k')
    plt.xlabel("Score")
    plt.ylabel("Frequency")
    plt.title("Histogram of Scores")
    plt.show()

In [9]:
# Optionally, save the cleaned dataframe to disk as a new parquet file or CSV
df.to_parquet("cleaned_human_face_emotions.parquet")
# Alternatively, you can save as CSV:
# df.to_csv("cleaned_human_face_emotions.csv", index=False)

# drop qa column
df.drop(columns=["qa"], inplace=True)

print(df.head())

                                               image  emotion
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      sad
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...    anger
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  neutral
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...     fear
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  content


Now we just have images in the first column with the emotion in the second column.

In [12]:
from sklearn.model_selection import train_test_split

# Separate feature (X) and label (y)
X = df['image']
y = df['emotion']

# Perform a stratified split to keep class distribution consistent
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 80% training, 20% testing
    random_state=42,    # for reproducibility
    stratify=y          # important for classification
)

# Validation split from X_train if needed:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,     # 25% of the training set (which is 20% of the total) -> 15% overall
    random_state=42,
    stratify=y_train
)

print("Training set size:", len(X_train))
print("Test set size:", len(X_test))
print("Validation set size:", len(X_val))

Training set size: 5640
Test set size: 1880
Validation set size: 1880


In [13]:
# Image bytes -> numpy arrays
def decode_images(image_series, target_size=(224, 224)):
    """
    Takes a pandas Series of dictionaries, each containing {'bytes': ...}.
    Decodes them into a list of NumPy arrays (RGB).
    Resizes images to target_size.
    Normalizes pixel values to [0, 1].

    Returns:
      - A NumPy array of shape (num_samples, target_size[0], target_size[1], 3)
    """
    decoded_list = []
    for item in image_series:
        # item should be a dict like {'bytes': b'...'}
        try:
            img_bytes = item['bytes']
            with Image.open(io.BytesIO(img_bytes)) as img:
                # Convert to RGB if needed
                img = img.convert('RGB')
                # Resize
                img = img.resize(target_size)
                # Convert to array
                arr = np.array(img, dtype=np.float32) / 255.0
            decoded_list.append(arr)
        except Exception as e:
            # If there's a bad image, you might want to handle or skip it
            print("Error decoding image:", e)
            # Optionally skip or handle it somehow. For now, let's skip:
            # Continue with the loop
            continue

    return np.stack(decoded_list, axis=0)

print("\nDecoding and resizing images...")

# Decode train set
X_train_array = decode_images(X_train, target_size=(224, 224))
print("X_train_array shape:", X_train_array.shape)

# Decode val set
X_val_array = decode_images(X_val, target_size=(224, 224))
print("X_val_array shape:", X_val_array.shape)

# Decode test set
X_test_array = decode_images(X_test, target_size=(224, 224))
print("X_test_array shape:", X_test_array.shape)


Decoding and resizing images...
X_train_array shape: (5640, 224, 224, 3)
X_val_array shape: (1880, 224, 224, 3)
X_test_array shape: (1880, 224, 224, 3)


In [14]:
# Encode labels
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded   = label_encoder.transform(y_val)
y_test_encoded  = label_encoder.transform(y_test)

print("\nLabel classes found:", label_encoder.classes_)
print("Sample of encoded labels:", y_train_encoded[:10])


Label classes found: ['anger' 'content' 'disgust' 'fear' 'happy' 'neutral' 'sad' 'surprise']
Sample of encoded labels: [6 3 5 6 0 6 7 3 4 0]


In [16]:
pip install tensorflow

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached rich-13.9.4-py3-none-any.whl.metadata (18 kB)
  Using cached namex-0.0.8-py3-none-any.whl.metadata (246 bytes)
     ---------------------------------------- 0.0/50.1 kB ? eta -:--:--
     ---------------------------------------- 50.1/50.1 kB 2.7 MB/s eta 0:00:00
  Using cached Markdown-3.7-py3-none-any.whl.metadata (7.0 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached MarkupSafe-3.0.2-cp312-cp312-win_amd64.whl.metadata (4.1 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached mdur

In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Suppose you have: (224, 224, 3) images
# If you used a different size (e.g. 160x160 for MobileNet), be consistent

num_classes = len(label_encoder.classes_)

# 1) Load a MobileNetV2 (or EfficientNet, ResNet, etc.) without its top layers
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# 2) Freeze the base_model so we only train the new head first
base_model.trainable = False

# 3) Build your classifier on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# 4) Train the new top layers
history = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# 5) (Optional) Fine-tune deeper layers
# Unfreeze part (or all) of base_model and re-compile with a lower learning rate
base_model.trainable = True
# You can selectively unfreeze only some layers:
# for layer in base_model.layers[:100]:
#     layer.trainable = False

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(1e-5),  # smaller LR for fine-tuning
    metrics=['accuracy']
)

history_fine = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

test_loss, test_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nFinal Test Loss after fine-tuning: {test_loss:.4f}")
print(f"Final Test Accuracy after fine-tuning: {test_acc:.4f}")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │        10,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,268,232 (8.65 MB)

 Trainable params: 10,248 (40.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 58s 287ms/step - accuracy: 0.1972 - loss: 2.1586 - val_accuracy: 0.3298 - val_loss: 1.8116
Epoch 2/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 48s 271ms/step - accuracy: 0.3204 - loss: 1.7932 - val_accuracy: 0.3628 - val_loss: 1.7476
Epoch 3/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 47s 263ms/step - accuracy: 0.3705 - loss: 1.6515 - val_accuracy: 0.3750 - val_loss: 1.7099
Epoch 4/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 47s 266ms/step - accuracy: 0.4011 - loss: 1.6050 - val_accuracy: 0.3771 - val_loss: 1.7064
Epoch 5/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 47s 264ms/step - accuracy: 0.4421 - loss: 1.5294 - val_accuracy: 0.3681 - val_loss: 1.6982
59/59 ━━━━━━━━━━━━━━━━━━━━ 12s 207ms/step - accuracy: 0.3541 - loss: 1.7879

Test Loss: 1.7301
Test Accuracy: 0.3739
Epoch 1/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 248s 1s/step - accuracy: 0.2521 - loss: 2.1557 - val_accuracy: 0.3351 - val_loss: 1.9583
Epoch 2/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 194s 1s/step - accuracy: 0.3762 - loss: 1.6960 - val_accuracy: 0.3

# Use transfer learning with pre-trained CNN model
convert the existing pipeline to use a pre-trained network such as ResNet50 in a transfer‐learning setup for multi-class classification using cross-entropy loss. In a transfer-learning approach, it typically replace the top (classification) layers of the pre-trained network with our own custom head and use a loss such as categorical cross-entropy (or sparse categorical cross-entropy if your labels remain as integers). We then train the added head first, and optionally fine-tune the deeper layers later.

In [18]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Assume your decoded image arrays and encoded labels are available as:
# X_train_array, X_val_array, X_test_array and y_train_encoded, y_val_encoded, y_test_encoded.
# And 'num_classes' is defined, e.g., num_classes = len(label_encoder.classes_)

# Pick one of the pre-trained models; here we use ResNet50.
base_model = tf.keras.applications.ResNet50(
    input_shape=(224, 224, 3),
    include_top=False,  # Remove the default classification head
    weights='imagenet'
)

# Freeze the base model to train only the new classification head initially.
base_model.trainable = False

# Build a new model on top of the base model.
model = models.Sequential([
    base_model,
    # Global average pooling to reduce spatial dimensions.
    layers.GlobalAveragePooling2D(),
    # Optional dropout for regularization.
    layers.Dropout(0.2),
    # Final Dense layer for multi-class classification using softmax.
    layers.Dense(num_classes, activation='softmax')
])

# When your labels are provided as integers, you can use sparse categorical crossentropy.
# Alternatively, if you one-hot encode your labels, you can use tf.keras.losses.categorical_crossentropy.
model.compile(
    loss='sparse_categorical_crossentropy',  
    optimizer='adam',
    metrics=['accuracy']
)

# Show the model summary.
model.summary()

# Train the new head.
history = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

# Evaluate on the test set.
test_loss, test_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Optional: Fine-tuning
# unfreeze all layers
base_model.trainable = True

# Freeze all layers except the last two.
for layer in base_model.layers[:-2]:
    layer.trainable = False


# It is generally recommended to use a lower learning rate for fine-tuning.
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(1e-5),
    metrics=['accuracy']
)

# Continue training (fine-tuning).
history_fine = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

final_loss, final_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nFinal Test Loss after fine-tuning: {final_loss:.4f}")
print(f"Final Test Accuracy after fine-tuning: {final_acc:.4f}")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │        16,392 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,604,104 (90.04 MB)

 Trainable params: 16,392 (64.03 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 228s 1s/step - accuracy: 0.1380 - loss: 2.1877 - val_accuracy: 0.1415 - val_loss: 2.0692
Epoch 2/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 221s 1s/step - accuracy: 0.1537 - loss: 2.0913 - val_accuracy: 0.1718 - val_loss: 2.0600
Epoch 3/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - accuracy: 0.1402 - loss: 2.0853 - val_accuracy: 0.1638 - val_loss: 2.0752
Epoch 4/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 213s 1s/step - accuracy: 0.1663 - loss: 2.0633 - val_accuracy: 0.1979 - val_loss: 2.0394
Epoch 5/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 212s 1s/step - accuracy: 0.1697 - loss: 2.0529 - val_accuracy: 0.1750 - val_loss: 2.0500
59/59 ━━━━━━━━━━━━━━━━━━━━ 54s 921ms/step - accuracy: 0.1795 - loss: 2.0507

Test Loss: 2.0493
Test Accuracy: 0.1809
Epoch 1/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - accuracy: 0.1675 - loss: 2.0526 - val_accuracy: 0.1793 - val_loss: 2.0280
Epoch 2/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 217s 1s/step - accuracy: 0.1917 - loss: 2.0287 - val_accuracy: 0.2011 - val_

In [23]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Again assume: X_train_array, X_val_array, X_test_array, and y_train_encoded, y_val_encoded, y_test_encoded.
# And num_classes is defined.

# Use VGG16 as the base pre-trained model.
base_model = tf.keras.applications.VGG16(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze the base model initially.
base_model.trainable = False

# Build the model.
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),  # VGG16 sometimes benefits from a higher dropout.
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model with sparse categorical cross entropy loss.
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# Train the classifier head.
history = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

# Evaluate on test data.
test_loss, test_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Fine-tuning:
# Unfreeze all first
base_model.trainable = True

# Freeze all layers except the last two.
for layer in base_model.layers[:-2]:
    layer.trainable = False


model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(1e-5),
    metrics=['accuracy']
)

history_fine = model.fit(
    X_train_array, y_train_encoded,
    validation_data=(X_val_array, y_val_encoded),
    epochs=5,
    batch_size=32
)

final_loss, final_acc = model.evaluate(X_test_array, y_test_encoded)
print(f"\nFinal Test Loss after fine-tuning: {final_loss:.4f}")
print(f"Final Test Accuracy after fine-tuning: {final_acc:.4f}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │         4,104 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,718,792 (56.15 MB)

 Trainable params: 4,104 (16.03 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 870s 5s/step - accuracy: 0.1392 - loss: 2.2689 - val_accuracy: 0.2271 - val_loss: 2.0108
Epoch 2/5
177/177 ━━━━━━━━━━━━━━━━━━━━ 513s 3s/step - accuracy: 0.1850 - loss: 2.0492 - val_accuracy: 0.2963 - val_loss: 1.9532
Epoch 3/5
123/177 ━━━━━━━━━━━━━━━━━━━━ 1:54 2s/step - accuracy: 0.2040 - loss: 1.9989

KeyboardInterrupt: 

# Grad CAM Explainability
compute and display Grad-CAM heatmaps that highlight image regions contributing to a specific classification.

* Build a gradient model that extracts the output of a chosen convolutional layer (normally the last one) along with the model predictions
* Compute the gradients of the target class score with respect to that convolutional output
* Aggregate these gradients (global-average pooling over the spatial dimensions) to obtain weights
* Compute a heatmap as a weighted combination of the convolutional feature maps
* Overlay the heatmap back on the original image

In [ ]:
# pip install opencv-python

   ---------------------------------------- 0.0/39.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/39.5 MB 1.3 MB/s eta 0:00:31
   ---------------------------------------- 0.2/39.5 MB 2.3 MB/s eta 0:00:18
    --------------------------------------- 0.7/39.5 MB 6.0 MB/s eta 0:00:07
   - -------------------------------------- 1.5/39.5 MB 9.3 MB/s eta 0:00:05
   -- ------------------------------------- 2.3/39.5 MB 11.5 MB/s eta 0:00:04
   --- ------------------------------------ 3.2/39.5 MB 13.7 MB/s eta 0:00:03
   ---- ----------------------------------- 4.1/39.5 MB 14.6 MB/s eta 0:00:03
   ---- ----------------------------------- 4.4/39.5 MB 14.8 MB/s eta 0:00:03
   ---- ----------------------------------- 4.4/39.5 MB 14.8 MB/s eta 0:00:03
   ----- ---------------------------------- 5.1/39.5 MB 12.1 MB/s eta 0:00:03
   ------ --------------------------------- 6.1/39.5 MB 13.4 MB/s eta 0:00:03
   ------- -------------------------------- 7.2/39.5 MB 14.5 MB/s eta 0:00:0

In [22]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Computes a Grad-CAM heatmap for a given image and model.
    
    Args:
      img_array: Preprocessed image array with shape (1, height, width, channels).
      model: The trained model.
      last_conv_layer_name: Name of the convolutional layer to use for Grad-CAM.
      pred_index: (Optional) index of the target class. If None, the model prediction is used.
      
    Returns:
      heatmap: A numpy 2D array of the heatmap.
    """
    # Create a model that maps the input image to the activations
    # of the last conv layer as well as the final predictions.
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    # Record operations for automatic differentiation.
    # calculate the loss as the score for the target class 
    # (or the predicted class if pred_index is not provided)
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        # Get the score for target class.
        loss = predictions[:, pred_index]
    
    # Compute the gradient of the target class output with respect to the feature map.
    grads = tape.gradient(loss, conv_outputs)
    
    # Pool the gradients over all the axes leaving out the channel dimension.
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Get the convolution outputs for the first (and only) image.
    conv_outputs = conv_outputs[0]
    
    # Weight the output feature map channels by the corresponding gradients.
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Apply ReLU (only keep positive activations) and normalize the heatmap.
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def overlay_heatmap_on_image(image, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Overlays the heatmap onto the original image.
    
    Args:
      image: Original image as a numpy array with shape (height, width, 3) in RGB.
      heatmap: 2D numpy array containing the heatmap (values between 0 and 1).
      alpha: Transparency factor for the heatmap overlay.
      colormap: OpenCV colormap to apply.
      
    Returns:
      overlayed_image: The image with the heatmap overlay.
    """
    # Resize heatmap to match the image size
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    # Convert the heatmap to 0-255 scale.
    heatmap = np.uint8(255 * heatmap)
    # Apply the chosen colormap.
    heatmap = cv2.applyColorMap(heatmap, colormap)
    # OpenCV uses BGR by default, so convert if your image is in RGB.
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    # Superimpose the heatmap onto the original image.
    overlayed_image = cv2.addWeighted(image, 1 - alpha, heatmap, alpha, 0)
    return overlayed_image

# Example Usage:
# 1. Load and preprocess an image. Make sure it fits the input shape the model expects.
from tensorflow.keras.preprocessing import image as kp_image

def load_preprocess_image(img_path, target_size=(224, 224)):
    img = kp_image.load_img(img_path, target_size=target_size)
    img_array = kp_image.img_to_array(img)
    # For models like ResNet or VGG ensure proper preprocessing. e.g.,
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
    return img_array, np.array(img)

# Replace 'path_to_image.jpg' with the path to your image.
img_array, original_img = load_preprocess_image("path_to_image.jpg")

# 2. Choose the last convolutional layer from your model.
# For example, if using ResNet50 the last convolution layer name is 'conv5_block3_out'.
last_conv_layer_name = 'conv5_block3_out'  # Update this based on your model architecture.

# 3. Compute the Grad-CAM heatmap.
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

# 4. Overlay the heatmap on the original image.
overlayed_img = overlay_heatmap_on_image(original_img, heatmap, alpha=0.4)

# 5. Display the original image, heatmap, and the overlay.
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Original Image")
plt.imshow(original_img)
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title("Grad-CAM Heatmap")
plt.imshow(heatmap, cmap='jet')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title("Overlayed Image")
plt.imshow(overlayed_img)
plt.axis('off')

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'path_to_image.jpg'